In [1]:
import pandas as pd
import os
studies = ["Katzmarski12"]

for study in studies:
    print(study)
    answers_df = pd.read_csv(f"{study}/supplemental_files/task_answers.csv", keep_default_na= False)
    task_answer_dict = {}
    task_answers_txt = "task,answer,amount,answer_num\n"
    for i in range(len(answers_df)): 
        task = answers_df["task"].iloc[i]
        answer = answers_df["answer"].iloc[i]
        if study == "Ajami18": 
            part = answers_df["participant"].iloc[i]
            if part <= 12: continue
        if task not in task_answer_dict: 
            task_answer_dict[task] = {}
        if answer not in task_answer_dict[task]: 
            task_answer_dict[task][answer] = 0
        task_answer_dict[task][answer] = task_answer_dict[task][answer] + 1
    for task in task_answer_dict: 
        with open(f"{study}\\csv_files\\zero_shot\\{study}_zero_shot_{task}.csv") as f:
            task_txt = f.read()
        for answer_num, answer in enumerate(task_answer_dict[task]):
            answer_txt = ""
            csv_answer_Txt = str(answer)
            special_case = False
            match (study): #deal with special cases
                case "Ajami18": 
                    if str(answer) == "": 
                        answer_txt = task_txt.replace("<<Answer>>", "<<  >>")
                        special_case = True
                    elif str(answer)[0].isalpha():
                        if len(answer) > 1: 
                            a_txt = ""
                            for j, a in enumerate(answer): 
                                a_txt += f"<< {a} >>"
                                if j == len(answer)-1: 
                                    continue
                                a_txt += " "
                        answer_txt = task_txt.replace("<<Answer>>", a_txt)
                        special_case = True
                case "Bauer19": 
                    if answer[0] == "x": 
                        a_txt = ""
                        for i in range(len(answer)):
                            a_txt += f"<< {answer[i]} >>"
                            if i < len(answer): 
                                a_txt += " "
                            else: 
                                a_txt += "."
                        answer_txt = task_txt.replace("<<Answer>>", a_txt)
                        special_case = True
                case "Börstler16": 
                    letters = ["A", "B", "C", "D", "E", "F"]
                    answers = answer.split("#")
                    answer_txt = task_txt.replace("AnswerR1", answers[0])
                    answer_txt = answer_txt.replace("AnswerR2", answers[1])
                    for j,a in enumerate(answers[2:]): 
                        letter = letters[j]
                        new_a = a.replace("\"", "\"\"")
                        answer_txt = answer_txt.replace(f"<< Answer{letter} >>", f"<< {new_a} >>.")
                    special_case = True
                case "Gopstein17a": 
                    if "\"" in str(answer): 
                        a_txt = answer.replace("\"", "\"\"")
                        answer_txt = task_txt.replace("<<Answer>>", f"<< {a_txt} >>")
                        special_case = True
                case "Gopstein17b": 
                    answer_lines = answer.split("\r\n")
                    csv_answer_Txt = csv_answer_Txt.replace("\r\n", "\n")
                    a_txt = "\n"
                    for answer_line in answer_lines: 
                        a_txt += f" << {answer_line} >>.\n"
                    special_case = True 
                    answer_txt = task_txt.replace("<<Answer>>.", f"{a_txt}")
                case "Medeiros19": 
                    answer_dict = {"Totally positive": 1, "Positive": 2, "Neither negative or positive": 3, "Negative": 4, "Totally negative": 5} 
                    answer_id = answer_dict[answer]
                    answer_txt = task_txt.replace("<<Answer>>", f"<< {answer_id} >>")
                    csv_answer_Txt = str(answer_id)
                    special_case = True
                case "Peitek18b": 
                    if answer[0] == "X" and len(answer) > 1:
                        a_txt = ""
                        for i, a in enumerate(answer): 
                            if a != "X": print("WARNING")
                            a_txt += "<< X >>"
                            if i < len(answer)-1:
                                a_txt += " "
                        answer_txt =  task_txt.replace("<<Answer>>", a_txt)
                        special_case = True
            if not special_case:          
                answer_txt += task_txt.replace("<<Answer>>", f"<< {answer} >>")
            os.makedirs(f"{study}\\csv_files\\ablations\\no_order\\", exist_ok=True)
            csv_answer_Txt = csv_answer_Txt.replace("\"", "\"\"")
            task_answers_txt += f"{task},\"{csv_answer_Txt}\",{task_answer_dict[task][answer]},{answer_num}\n"
            with open(f"{study}\\csv_files\\ablations\\no_order\\{study}_no_order_{task}_{answer_num}.csv", "w") as f: 
                f.write(answer_txt)
    with open(f"{study}\\answer_amounts.csv", "w") as f:
        f.write(task_answers_txt)



Katzmarski12
